# ETF Cash-Secured Put Backtest

Feasibility study swapping bull put spreads on the 188-equity universe for **cash-secured puts on SPY, QQQ, IWM, GLD, TLT** ranked by VRP.

- **Forecast:** LogHAR, WF-retrained on the 5 ETFs (pooled) — `etf_predictions.parquet`
- **Signal:** `VRP = atm_iv^2 - y_pred`, rank by VRP within each day
- **Structure:** single short put, target delta -0.30, DTE [30, 60], monthly expirations
- **Capital:** $250,000 pretend (notional-reserving CSPs, 1 SPY CSP ≈ $50K)
- **Compare against:** Level 2 bull put spread result (Sharpe -0.68, NO BET verdict)

In [ ]:
import polars as pl, numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.stats import skew, kurtosis, norm

ROOT = Path('../data/processed/backtest')
tl = pl.read_parquet(ROOT / 'level2_csp_trade_log.parquet')
eq = pl.read_parquet(ROOT / 'level2_csp_daily_equity.parquet')
print(f'trades: {len(tl)}, days: {len(eq)}')

## 1. Headline metrics

In [ ]:
cap = eq['capital'].to_numpy()
daily_ret = (cap[1:] - cap[:-1]) / cap[:-1]
sr = daily_ret.mean() / daily_ret.std(ddof=1) * np.sqrt(252)

peaks = np.maximum.accumulate(cap)
mdd = float(((peaks - cap) / peaks).max())

skw = skew(daily_ret); kur = kurtosis(daily_ret, fisher=True)
n = len(daily_ret)
sr_se = np.sqrt((1 - skw*sr + kur/4*sr**2) / (n-1))
z = sr / sr_se
p_dsr = 1 - norm.cdf(z)

years = (eq['date'].max() - eq['date'].min()).days / 365.25
cagr = (cap[-1]/cap[0])**(1/years) - 1

print(f'Sharpe:           {sr:.2f}')
print(f'Deflated Sharpe z {z:.2f}, p(SR>0)={p_dsr:.4f}')
print(f'Max drawdown:     {mdd:.1%}')
print(f'CAGR on total:    {cagr:.2%}')
print(f'Win rate:         {(tl["net_pnl"]>0).mean():.1%}')
print(f'Mean net PnL:     ${tl["net_pnl"].mean():.2f}')
print(f'Skew/Excess kurt: {skw:+.2f} / {kur:+.2f}')

## 2. Verdict table — CSPs on ETFs vs. Bull Puts on equities

In [ ]:
# Bull-put verdict numbers are from project_backtest_sinclair_lopez_verdict.md
verdict = pl.DataFrame({
    'metric':        ['Sharpe', 'Deflated Sharpe p-val', 'Max DD', 'Kelly frac', 'Win rate', 'n trades', 'Verdict'],
    'bull_put_eq':   ['-0.68', '~1.00', '3%',  '-0.0114', 'n/a', '~150',   'NO BET'],
    'csp_etf':       [f'{sr:+.2f}', f'{p_dsr:.4f}', f'{mdd:.1%}', 'tbd', f'{(tl["net_pnl"]>0).mean():.1%}', f'{len(tl)}', 'PASS Sharpe gate'],
})
print(verdict)

## 3. Equity curve

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
dates = eq['date'].to_numpy()
a1.plot(dates, cap, lw=1.3, label='capital')
a1.plot(dates, peaks, lw=0.8, ls='--', color='grey', label='peak')
a1.set_ylabel('capital ($)')
a1.legend(); a1.grid(alpha=0.3)
a1.set_title(f'ETF CSP backtest: Sharpe={sr:.2f}, MDD={mdd:.1%}, CAGR={cagr:.1%}')

a2.fill_between(dates, eq['notional_reserved'], alpha=0.4, label='reserved')
a2.set_ylabel('reserved cash ($)')
a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Per-symbol breakdown

Where does the PnL come from? If 1-2 ETFs carry the whole result, the signal isn't generalizing.

In [ ]:
per_sym = (tl.group_by('symbol').agg(
    pl.len().alias('n'),
    (pl.col('net_pnl')>0).mean().alias('win_rate'),
    pl.col('net_pnl').mean().alias('mean_pnl'),
    pl.col('net_pnl').sum().alias('total_pnl'),
    pl.col('net_pnl').min().alias('worst'),
).sort('total_pnl', descending=True))
print(per_sym)

## 5. Exit reason breakdown

In [ ]:
print(tl.group_by('exit_reason').agg(
    pl.len().alias('n'),
    pl.col('net_pnl').mean().alias('mean'),
    pl.col('net_pnl').sum().alias('total'),
).sort('n', descending=True))

## 6. PnL distribution

Watch for fat left tail (breaches below strike with no hedge leg).

In [ ]:
pnl = tl['net_pnl'].to_numpy()
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(pnl, bins=50, edgecolor='white')
ax.axvline(0, color='black', lw=1)
ax.axvline(pnl.mean(), color='red', ls='--', lw=1, label=f'mean ${pnl.mean():+.0f}')
ax.set_xlabel('net PnL ($)'); ax.set_ylabel('count'); ax.legend()
ax.set_title('Per-trade PnL distribution')
plt.show()

for q in [0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]:
    print(f'  q{int(q*100):02d}: ${np.quantile(pnl, q):+,.0f}')

## 7. Caveats

1. **Capital base inflated to $250K** — one SPY CSP reserves $50K. Not deployable at EUR 10K.
2. **Average utilization ~39%**. Return on deployed capital is ~3x the headline CAGR, but idle cash must sit in money market.
3. **Concentration**: QQQ+SPY drove almost all PnL (see §4). TLT and GLD are noise on this sample.
4. **Excess kurtosis 12.3** — fat tails even with defined max loss at strike=0. Single worst trade matters.
5. **Sample size**: 164 trades over 3 years. DSR is significant but daily-return autocorrelation from carry inflates z.
6. **LogHAR only**. No LightGBM retraining for ETFs; a stronger forecast may or may not improve the signal — in the equity case it didn't help much vs LogHAR.

The headline result (Sharpe ~1.7, MDD ~2%, CAGR ~1.7%) is a meaningful change from the bull-put NO BET. It does **not** automatically mean the strategy is deployable — it means the vehicle swap resolves the tail-risk breach problem that killed bull puts, and the VRP signal carries real information on index ETFs.